# GLiNER 1 Evaluation

Links used for help:
1. https://colab.research.google.com/drive/1HNKd74cmfS9tGvWrKeIjSxBt01QQS7bq?usp=sharing (The author's finetuning notebook)
2. Claude for trouble shooting

## ZERO SHOT

In [28]:
FILE_PATH = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/my_data/final_dataset_1st_may.conll"


In [ ]:
pip install gliner

In [29]:
from collections import defaultdict
import random
import json
from gliner import GLiNER
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2
import os
import torch
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import SpanDataCollator

In [ ]:
pip install --upgrade huggingface_hub

In [30]:
def load_conll(file):
    sentences = []
    current_sentence = []

    with open(file, "r", encoding = "utf-8") as infile:
        for line in infile:
            line = line.strip()

            if line == "":
                if len(current_sentence)> 0:
                    sentences.append(current_sentence)
                    current_sentence = []
            elif line.startswith("-DOCSTART-"):
                continue

            else:
                parts = line.split("\t")

                if len(parts) == 2:
                    token = parts[0]
                    label = parts[1]
                    current_sentence.append((token, label))
                else:
                    print(f"missed line!{repr(line)} ")

        if len(current_sentence) > 0:
            sentences.append(current_sentence)
    return sentences

In [31]:
def get_all_labels_in_sentence(sentence):
    entity_types = set()
    for token, label in sentence:
        if label.startswith("B-"):
            entity_types.add(label.split("-", 1)[1])
    return entity_types

In [ ]:
gliner1_model= "urchade/gliner_large-v2.1"
threshold = 0.5
# labels given to gliner
gliner_labels = ["Person", "Organization", "Country", "Law", "Date", "Tax_Concept", "Tax_Type", "Provision", "Jurisdiction", "Court"]
#labels we want mapped back to the target set
label_mapping = {"Person": "PERSON", "Organization": "ORG", "Country": "GPE", "Law": "LAW", "Date": "DATE", "Tax_Concept": "TAX_CONCEPT","Tax_Type": "TAX_TYPE", "Provision": "PROVISION", "Jurisdiction": "JURISDICTION",
    "Court": "COURT", }

def get_token_offsets(sentence, tokens):
    """
    this returns the offsets for each token
    """
    offsets  = []
    position = 0
    for token in tokens: #go through every token
        while position < len(sentence) and sentence[position] == " ": #skipping spaces before the next token
            position += 1
        start = position #starting character position
        end = start + len(token) #get the ending characte position
        offsets.append((start, end)) #save them
        position = end #move position to end of the current token

    return offsets

def spans_to_bio(tokens, sentence_string, spans):
    """
    this converts GLiNER character-level spans into IOB2 labels
    """
    bio_sequence  = ["O"] * len(tokens) #start with all tokens as O!
    token_offsets = get_token_offsets(sentence_string, tokens) #get character positions of each token
    #sort the predicted spans from highest to lowest
    sorted_spans = sorted(spans, key=lambda span: span["score"], reverse=True)
    
    for entity in sorted_spans: #for every predicted span,
            mapped_label = label_mapping.get(entity["label"], entity["label"]) #map the gliner label to the target label
            entity_started = False # keep track of whether the the entity started
            for token_index, (token_start, token_end) in enumerate(token_offsets): #go thru each token and its char offsets
                if token_start >= entity["start"] and token_end <= entity["end"]:  #check if token is inside predicted entity
                    if bio_sequence[token_index] == "O": #only assign to unlabeled tokens
                        if entity_started:
                            bio_sequence[token_index] = f"I-{mapped_label}" #if its an already started entity, its I-
                        else:
                            bio_sequence[token_index] = f"B-{mapped_label}" #otherwise its the beginning of a new entity!
                            entity_started = True

    return bio_sequence

def predict_with_gliner(model, test_sentences, gliner_labels, threshold=0.5, pred_conll_path=None):
    """
    this runs gliner zero shot prediction on every sentence inputted and returns the gold and pred labels
    """
    gold_sequences = []
    pred_sequences = []
    conll_lines = []

    for sentence_index, sentence in enumerate(test_sentences): #for every sentence in the test set,
        tokens = []
        gold_bio = []
        
        for token, label in sentence:
            tokens.append(token) #get a list of tokens
            gold_bio.append(label) #and a list of gold labels
        sentence_string = " ".join(tokens) #join back as one sentence for prediction
        spans = model.predict_entities(sentence_string, gliner_labels, threshold=threshold, flat_ner=True) #predict with gliner!
        pred_bio = spans_to_bio(tokens, sentence_string, spans) #convert the char predictions to bio labels
        gold_sequences.append(gold_bio) #add the gold labels back
        pred_sequences.append(pred_bio) #add the pred labels back

        for token, pred in zip(tokens, pred_bio):#we can write the preds to conll now
            conll_lines.append(f"{token}\t{pred}") 
        conll_lines.append("")

        if (sentence_index + 1) % 50 == 0: #tracker since it takes a while to run
            print(f"  Predicted {sentence_index + 1}/{len(test_sentences)} sentences...")

    if pred_conll_path:
        with open(pred_conll_path, "w", encoding="utf-8") as f:
            f.write("\n".join(conll_lines))
        print(f"Predictions saved to: {pred_conll_path}")

    return gold_sequences, pred_sequences


def evaluate(gold_sequences, pred_sequences):
    """
    evaluate gliner's predictions using seqeval strict span-level evaluation
    """
    #now we use strict seqeval for eval!
    report = classification_report(gold_sequences,pred_sequences, scheme=IOB2, mode="strict",output_dict=False,zero_division=0)
    print(report)

In [21]:
sentences = load_conll(FILE_PATH)
print(f"Train: {len(sentences)} sentences")

Train: 656 sentences


In [ ]:
#GLiNER -- zero shot
model = GLiNER.from_pretrained(gliner1_model)
model.eval()
gold_sequences, pred_sequences = predict_with_gliner(
    model, sentences, gliner_labels, threshold=0.5,
    pred_conll_path=r"gliner_zeroshot_predictions.conll"
)
evaluate(gold_sequences, pred_sequences)

/opt/anaconda3/lib/python3.13/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/anaconda3/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/opt/anaconda3/lib/python3.13/sit

  Predicted 50/656 sentences...
  Predicted 100/656 sentences...
  Predicted 150/656 sentences...


/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 386 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  Predicted 200/656 sentences...
  Predicted 250/656 sentences...
  Predicted 300/656 sentences...
  Predicted 350/656 sentences...
  Predicted 400/656 sentences...
  Predicted 450/656 sentences...
  Predicted 500/656 sentences...
  Predicted 550/656 sentences...
  Predicted 600/656 sentences...
  Predicted 650/656 sentences...
Predictions saved to: TESTING_final_gliner_zeroshot_predictions.conll
              precision    recall  f1-score   support

       COURT       0.29      0.46      0.36       106
        DATE       0.72      0.79      0.75       132
         GPE       0.55      0.77      0.64       299
JURISDICTION       0.29      0.07      0.11       153
         LAW       0.13      0.62      0.21        65
         ORG       0.20      0.43      0.27       221
      PERSON       0.46      0.81      0.59       186
   PROVISION       0.09      0.18      0.12        87
 TAX_CONCEPT       0.32      0.33      0.32       353
    TAX_TYPE       0.35      0.62      0.45        86

   m

## FINE TUNING

In [ ]:
gliner1_model = "urchade/gliner_large-v2.1"
OUTPUT_DIR    = "gliner1_finetuned_on_opensource"
training_split = "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/inlegal_train_FINAL.conll"
validation_split = "/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/inlegal_validation_FINAL.conll"

def bio_to_spans(sentence):
    """this converst BIO labels into GLiNER entity spans."""
    tokens = []
    spans = []
    for token, label in sentence: # we get all the tokens from the sent
        tokens.append(token)
    current_start = None #start index of the current entity tracked
    current_label = None #entity type of the entity tracked
    for token_index, (token, label) in enumerate(sentence): #get each token's bio label
        if label.startswith("B-"): #if a new ent starts,
            if current_label is not None: #if another ent is being tracked, close it first
                spans.append([current_start, token_index - 1, current_label]) #save the start and end index and label!
            current_start = token_index #setting the start for the new entity
            current_label = label[2:] #remove bio prefix for ent type
        elif label.startswith("I-"): 
            entity_type = label[2:] #same here
            if current_label == entity_type: #if same ent type, continue ent
                continue
            if current_label is not None: #if another ent is tracked, close it first
                spans.append([current_start, token_index - 1, current_label]) #saving the previous ent
            current_start = token_index #start the new ent
            current_label = entity_type #and set the new ent type
        else:
            if current_label is not None: #if ent tracked, close it
                spans.append([current_start, token_index - 1, current_label]) #save the ent
                current_start = None #reset the ent start
                current_label = None #reset the ent type
    if current_label is not None: #if an ent continues until the end, we close it
        spans.append([current_start, len(tokens) - 1, current_label]) #using the final token as end index
    return tokens, spans

def sentences2glinerdict(sentences):
    """this converts a list of CoNLL sentences to a list of GLiNER-format dicts."""
    gliner_data = []

    for sentence in sentences:
        tokens, spans = bio_to_spans(sentence) #convert the bio labels into spans
        gliner_data.append({"tokenized_text": tokens, "ner": spans}) #save tokens and spans for special gliner formtting
    return gliner_data

def finetune_gliner(train_gliner_data, val_gliner_data, model_name, output_dir):
    """this fine-tunes a GLiNER model on the provided training data."""
    os.makedirs(output_dir, exist_ok=True)

    print(f"Loading base model: {model_name}")
    model = GLiNER.from_pretrained(model_name)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=5e-6,
        weight_decay=0.01,
        others_lr=1e-5,
        others_weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=6,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=67,
        # fp16=fp16,
    )

    data_collator = SpanDataCollator(
        config=model.config,
        data_processor=model.data_processor,
        prepare_labels=True,
        prepare_entities=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_gliner_data,
        eval_dataset=val_gliner_data,
        processing_class=model.data_processor.transformer_tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    return model

sentences = load_conll(FILE_PATH)
train_sentences = load_conll(training_split)
val_sentences = load_conll(validation_split)
train_gliner = sentences2glinerdict(train_sentences)
val_gliner = sentences2glinerdict(val_sentences)

finetuned_model = finetune_gliner(
    train_gliner, val_gliner, gliner1_model, OUTPUT_DIR
)
# finetuned_model.eval()

# gold_sequences, pred_sequences = predict_with_gliner(
#     finetuned_model, test_sentences, gliner_labels, threshold=threshold
# )

# evaluate(gold_sequences, pred_sequences)

Loading base model: urchade/gliner_large-v2.1


/opt/anaconda3/lib/python3.13/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


KeyboardInterrupt: 

## EVALUATION

In [ ]:
#GLiNER -- existing legal data
gliner1_model = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/GLiNER_OS/model_output/checkpoint-5449"
sentences = load_conll(FILE_PATH)
print(f"Predicting {len(sentences)} sentences")
model = GLiNER.from_pretrained(gliner1_model)
model.eval()
gold_sequences, pred_sequences = predict_with_gliner(model, sentences, gliner_labels, threshold=0.5,
    pred_conll_path=r"gliner_OS_predictions.conll")
evaluate(gold_sequences, pred_sequences)

Predicting 656 sentences


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 537 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  Predicted 50/656 sentences...
  Predicted 100/656 sentences...
  Predicted 150/656 sentences...


/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 386 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  Predicted 200/656 sentences...
  Predicted 250/656 sentences...
  Predicted 300/656 sentences...
  Predicted 350/656 sentences...
  Predicted 400/656 sentences...
  Predicted 450/656 sentences...
  Predicted 500/656 sentences...
  Predicted 550/656 sentences...
  Predicted 600/656 sentences...
  Predicted 650/656 sentences...
Predictions saved to: TESTING_final_gliner_OS_predictions.conll
              precision    recall  f1-score   support

       COURT       0.64      0.69      0.66       106
        DATE       0.79      0.39      0.53       132
         GPE       0.65      0.69      0.67       299
JURISDICTION       0.40      0.16      0.23       153
         LAW       0.18      0.66      0.28        65
         ORG       0.44      0.35      0.39       221
      PERSON       0.80      0.69      0.74       186
   PROVISION       0.08      0.15      0.11        87
 TAX_CONCEPT       0.42      0.22      0.29       353
    TAX_TYPE       0.55      0.49      0.52        86

   micro a

In [ ]:
#GLiNER -- LLM labeled data
gliner1_model = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/GLiNER_LLM/model_output/checkpoint-3564 - Copy"
sentences = load_conll(FILE_PATH)
print(f"Predicting {len(sentences)} sentences")
model = GLiNER.from_pretrained(gliner1_model)
model.eval()
gold_sequences, pred_sequences = predict_with_gliner(model, sentences, gliner_labels, threshold=0.5,
    pred_conll_path=r"gliner_LLM_predictions.conll")
evaluate(gold_sequences, pred_sequences)

Predicting 656 sentences


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  Predicted 50/656 sentences...
  Predicted 100/656 sentences...
  Predicted 150/656 sentences...
  Predicted 200/656 sentences...
  Predicted 250/656 sentences...
  Predicted 300/656 sentences...
  Predicted 350/656 sentences...
  Predicted 400/656 sentences...
  Predicted 450/656 sentences...
  Predicted 500/656 sentences...
  Predicted 550/656 sentences...
  Predicted 600/656 sentences...
  Predicted 650/656 sentences...
Predictions saved to: 00_ERROR_ANALYSIS_PREDS_TESTING_123_final_gliner_LLM_predictions.conll
              precision    recall  f1-score   support

       COURT       0.43      0.74      0.55       106
        DATE       0.69      0.82      0.75       132
         GPE       0.71      0.84      0.77       299
JURISDICTION       0.54      0.85      0.66       153
         LAW       0.20      0.49      0.28        65
         ORG       0.39      0.49      0.43       221
      PERSON       0.49      0.81      0.61       186
   PROVISION       0.35      0.51      0.42   

In [ ]:
#GLiNER -- Mixed data
gliner1_model = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/GLiNER_MIXED/model_output/checkpoint-15624"
sentences = load_conll(FILE_PATH)
print(f"Predicting {len(sentences)} sentences")
model = GLiNER.from_pretrained(gliner1_model)
model.eval()
gold_sequences, pred_sequences = predict_with_gliner(model, sentences, gliner_labels, threshold=0.5,
    pred_conll_path=r"gliner_mixed_predictions.conll")
evaluate(gold_sequences, pred_sequences)

Predicting 656 sentences


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  Predicted 50/656 sentences...
  Predicted 100/656 sentences...
  Predicted 150/656 sentences...
  Predicted 200/656 sentences...
  Predicted 250/656 sentences...
  Predicted 300/656 sentences...
  Predicted 350/656 sentences...
  Predicted 400/656 sentences...
  Predicted 450/656 sentences...
  Predicted 500/656 sentences...
  Predicted 550/656 sentences...
  Predicted 600/656 sentences...
  Predicted 650/656 sentences...
Predictions saved to: TESTING_final_gliner_mixed_predictions.conll
              precision    recall  f1-score   support

       COURT       0.40      0.79      0.53       106
        DATE       0.62      0.86      0.72       132
         GPE       0.68      0.83      0.75       299
JURISDICTION       0.51      0.82      0.62       153
         LAW       0.17      0.54      0.26        65
         ORG       0.43      0.49      0.46       221
      PERSON       0.40      0.51      0.45       186
   PROVISION       0.31      0.49      0.38        87
 TAX_CONCEPT      

'              precision    recall  f1-score   support\n\n       COURT       0.40      0.79      0.53       106\n        DATE       0.62      0.86      0.72       132\n         GPE       0.68      0.83      0.75       299\nJURISDICTION       0.51      0.82      0.62       153\n         LAW       0.17      0.54      0.26        65\n         ORG       0.43      0.49      0.46       221\n      PERSON       0.40      0.51      0.45       186\n   PROVISION       0.31      0.49      0.38        87\n TAX_CONCEPT       0.29      0.36      0.32       353\n    TAX_TYPE       0.43      0.62      0.51        86\n\n   micro avg       0.43      0.61      0.51      1688\n   macro avg       0.42      0.63      0.50      1688\nweighted avg       0.45      0.61      0.51      1688\n'